# Attention language model

Start from the same Tiny Shakespeare token stream as the bigram model, then inspect token embeddings before adding attention.

In [1]:
import random
import sys
from pathlib import Path

import torch
from torch import nn

repo_root = Path.cwd()
if not (repo_root / "data").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.dataset import get_batch, load_tiny_shakespeare_tokens, split_token_stream

## Prepare token streams and batches

In [2]:
tokens, vocab, merges = load_tiny_shakespeare_tokens(repo_root / "data")
train_tokens, validation_tokens = split_token_stream(tokens)

vocab_size = len(vocab)
block_size = 8
batch_size = 32
n_embd = 32
head_size = 16

random.seed(42)
x_batch, y_batch = get_batch(
    "train", train_tokens, validation_tokens, block_size, batch_size
)
x_batch = torch.tensor(x_batch, dtype=torch.long)
y_batch = torch.tensor(y_batch, dtype=torch.long)

## Model scaffold

In [3]:
class AttentionLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd):
        super().__init__()
        self.head = self.Head(n_embd, head_size) # (B, T, H)
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.lm_head = nn.Linear(head_size, vocab_size)

    def forward(self, idx):
        x = self.token_embedding_table(idx)  # (B, T, C)
        x = self.head(x)  # (B, T, H)
        return self.lm_head(x) # (B, T, V)

    # # x: [B, T, C]
    # q,k,v: [B, T, H]
    # out: [B, T, H]
    class Head(nn.Module):
        def __init__(self, n_embd, head_size):
            super().__init__()
            self.n_embd = n_embd
            self.head_size = head_size
            self.query = nn.Linear(n_embd, head_size)
            self.key = nn.Linear(n_embd, head_size)
            self.value = nn.Linear(n_embd, head_size)

        def forward(self, x):
            B, T, C = x.shape   
            q = self.query(x)  # (B, T, H)
            k = self.key(x)  # (B, T, H)
            v = self.value(x)  # (B, T, H)
            attention = self.attention(q, k, v)  # (B, T, H)
            return attention

        def attention(self, q, k, v):
            # Compute attention scores
            attn_scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_size ** 0.5)  # (B, T, T)
            # Apply masking to attention scores
            mask = torch.triu(torch.ones(attn_scores.shape[:2], dtype=torch.bool, device=attn_scores.device))
            attn_scores = attn_scores.masked_fill(mask, -float('inf'))
            # Normalize attention scores
            attn_weights = torch.softmax(attn_scores, dim=-1)  # (B, T, T)
            # Compute attention outputs
            out = torch.matmul(attn_weights, v)  # (B, T, H)
            return out


model = AttentionLanguageModel(vocab_size, n_embd)
head = model.Head(n_embd, head_size)
token_embeddings = model.token_embedding_table(x_batch)
print(f"Token embedding shape (B, T, C): {tuple(token_embeddings.shape)}")

Token embedding shape (B, T, C): (32, 8, 32)


In [4]:
# Positional embeddings
positional_embeddings = nn.Parameter(torch.zeros(1, block_size, n_embd))
# Add positional embeddings to token embeddings
token_embeddings += positional_embeddings
print(f"Token embedding shape after adding positional embeddings (B, T, C): {tuple(token_embeddings.shape)}")

Token embedding shape after adding positional embeddings (B, T, C): (32, 8, 32)


In [5]:
# Attention


In [6]:
# Causal self-attention
import math
import torch
from torch import nn

key = nn.Linear(n_embd, head_size, bias=False)
query = nn.Linear(n_embd, head_size, bias=False)
value = nn.Linear(n_embd, head_size, bias=False)

# x       [B, T, C]
# q,k,v   [B, T, H]
# scores  [B, T, T]
# weights [B, T, T]
# out     [B, T, H]
# MATCH THE SHAPES OF THE LINEAR LAYERS!! 
def causal_self_attention(x, mask=None):
    k = key(x)
    q = query(x)
    v = value(x)

    scores = q @ k.transpose(-1, -2) / math.sqrt(head_size)
    scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    attention_out = weights @ v
    return attention_out

x = torch.randn(1, 8, n_embd)
mask = torch.ones(1, 8, 8)
causal_self_attention(x, mask)

# multiple heads


tensor([[[ 0.1529, -0.1199, -0.4763,  0.0199,  0.2319, -0.0942,  0.1836,
           0.1423,  0.0357, -0.2301, -0.2794, -0.1684, -0.0930,  0.1878,
           0.0895,  0.2136],
         [ 0.2021, -0.1885, -0.4230,  0.0491,  0.2338, -0.0526,  0.1869,
           0.0694,  0.1425, -0.1376, -0.2282, -0.1347, -0.0354,  0.1031,
           0.0612,  0.1878],
         [ 0.1948, -0.2065, -0.4023,  0.0416,  0.2288, -0.0536,  0.1720,
           0.0959,  0.1317, -0.1148, -0.2626, -0.1639, -0.0606,  0.1474,
           0.0652,  0.1916],
         [ 0.2210, -0.2749, -0.4080,  0.0460,  0.3070, -0.0176,  0.1620,
           0.0157,  0.2570, -0.0760, -0.2964, -0.1592, -0.0467,  0.0894,
           0.0977,  0.1999],
         [ 0.0908, -0.1498, -0.4259,  0.0446,  0.1495, -0.0918,  0.3018,
           0.0359,  0.1737, -0.2039, -0.2117, -0.1671, -0.0415, -0.0039,
           0.0637,  0.0900],
         [ 0.1948, -0.2776, -0.4305,  0.1106,  0.0044, -0.1208,  0.2505,
           0.0928,  0.4079, -0.1334, -0.2209, -0.216